<a href="https://colab.research.google.com/github/TorbjornLarsson/SCDA/blob/main/Lab3_P3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab exercise 3: Eigenvalues and SVD

## Part 3: Computation of eigenvalues

In the previous part of the lab, we learned how to calculate the eigenvalues and eigenvectors of a matrix in Python. But what methods are used under the hood? In this part we look at two numerical methods used for eigenvalue computations. From mathematics, you know that eigenvalues can be computed via the characteristic equation  𝑑𝑒𝑡(𝐴−𝜆𝐼)=0 . This is not an option in software though, since the coefficients of the characteristic equation can't be computed from determinant evaluations in a numerically stable way. It leads to large perturbations in the roots. In this lab, we will have a look at some computational methods to calculate the eigenvalues of a matrix.

### The Power method

The simplest numerical method for eigenvalue computations is the Power Method. We outline the algorithm below:<br>
<br>
Given a matrix $A$,
- Start with an initial eigenvector $v^{(0)}$, which is a guess, input by the user
- Calculate $v^{(1)}=Av^{(0)}, v^{(2)} = Av^{(1)}, v^{(3)} = Av^{(2)}, \ldots$, and so on. The sequence of $v^{(i)}$'s are normalized throughout the process

Using this calculation process, it can be shown that $v^{(i)}$ approaches one eigenvector of the matrix $A$. We terminate the process when we have reached a vector $v^{(i)}$ that is close enough to the true eigenvector (when the difference between two consecutive $v^{(i)}$ is smaller than a given tolerance). When the eigenvector is found, the eigenvalue can be computed with the formula $\lambda=\frac{v^TAv}{v^Tv}$ (the denominator disappears when $v$ is normalized).

You will now implement the Power Method yourself following the instructions below:


_1.1) First, define the matrix_
$A = \left( \begin{array}{ccc}
1 & 2 & 0 \\
1 & 1 & 2 \\
1 & 3 & 1
\end{array} \right) $
_and check eigenvalues/eigenvectors using numpy._

In [1]:
import numpy as np

# 1.1) First, define the matrix  A=(1 2 0; 1 1 2; 1 3 1) and check eigenvalues/eigenvectors using numpy.
A = np.array([[1, 2, 0],
              [1, 1, 2],
              [1, 3, 1]])

# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)

print("Matrix A:")
print(A)
print("\nEigenvalues:")
print(eigenvalues)
print("\nEigenvectors:")
print(eigenvectors)

Matrix A:
[[1 2 0]
 [1 1 2]
 [1 3 1]]

Eigenvalues:
[ 4.05137424  0.48269595 -1.5340702 ]

Eigenvectors:
[[-0.38725131 -0.89283019  0.51116045]
 [-0.59082433  0.23093234 -0.64765823]
 [-0.70778742  0.38668398  0.56502549]]


_1.2) Now implement the Power Method (with a for-loop so you can choose number of iterations). You can find a pseudocode for the implementation below. Make sure you understand how the pseudocode corresponds to the algorithm presented above, and then use it to write an actual Python implementation._
```
  v0 = ...  # initial eigenvector (initial guess), you can use v0=(1, 1,...,1)
  Normalize v0
  N = ...   # Number of iterations
  for k in range(N):
     v = Av0
     v = v/norm(v)  # Normalize v
     e = v.T(Av)    # Compute eigenvalue
     v0 = v
```
_1.3) When your code works, run it for different number of iterations ${\tt N}$. You can for example start with 5 iterations and then increase until you have for example 6 correct decimal places (compared with the true eigenvalues calculated above)._

In [2]:
import numpy as np

# 1.2) Implement the Power Method

# Initial eigenvector (initial guess), you can use v0=(1, 1,...,1)
v0 = np.ones(A.shape[1]) # A is already defined from previous cell

# Normalize v0
v0 = v0 / np.linalg.norm(v0)

N = 100 # Number of iterations (can be adjusted)

print(f"Initial v0: {v0}\n")

eigenvalue_estimate = 0

for k in range(N):
    v = A @ v0 # Calculate Av0

    # Normalize v
    norm_v = np.linalg.norm(v)
    v = v / norm_v

    # Compute eigenvalue (Rayleigh quotient for normalized vector)
    # Since v is normalized, v.T @ v = 1, so lambda = v.T @ A @ v
    e = v.T @ A @ v

    # Update v0 for the next iteration
    v0 = v

    # Optional: print progress to see convergence
    # if k % 10 == 0 or k == N-1:
    #    print(f"Iteration {k+1}: Eigenvalue = {e:.6f}")

print(f"\nPower Method Result after {N} iterations:")
print(f"Dominant Eigenvalue (e): {e:.6f}")
print(f"Corresponding Eigenvector (v): {v}")

# For comparison, let's get the dominant eigenvalue/eigenvector from numpy's calculation
# The dominant eigenvalue is the one with the largest absolute value
dominant_idx = np.argmax(np.abs(eigenvalues))
numpy_dominant_eigenvalue = eigenvalues[dominant_idx]
numpy_dominant_eigenvector = eigenvectors[:, dominant_idx]

print(f"\nNumpy's Dominant Eigenvalue: {numpy_dominant_eigenvalue:.6f}")
print(f"Numpy's Dominant Eigenvector: {numpy_dominant_eigenvector}")

# Check if the vectors are in the same direction (may differ by a sign)
# We can compare the absolute values or align the signs for better comparison
# For simplicity, let's just check the dot product of normalized vectors for direction
aligned_v = v * np.sign(numpy_dominant_eigenvector[0] / v[0]) if v[0] != 0 else v

print("\nIs the Power Method eigenvector close to Numpy's (after sign alignment)?")
print(f"Difference: {np.linalg.norm(aligned_v - numpy_dominant_eigenvector):.6f}")

Initial v0: [0.57735027 0.57735027 0.57735027]


Power Method Result after 100 iterations:
Dominant Eigenvalue (e): 4.051374
Corresponding Eigenvector (v): [0.38725131 0.59082433 0.70778742]

Numpy's Dominant Eigenvalue: 4.051374
Numpy's Dominant Eigenvector: [-0.38725131 -0.59082433 -0.70778742]

Is the Power Method eigenvector close to Numpy's (after sign alignment)?
Difference: 0.000000


From the results, answer the questions
- Which eigenvalue (and corresponding eigenvector) does the Power Method find?
- Is it exactly the same eigenvector compared with `numpy.linalg.eig`?

<br>
The Power method is an example of an <i>iterative method</i>, meaning that we get a sequence of solutions $\{\lambda_0, \lambda_1, \ldots \}$ (one solution each iteration), and that the sequence approaches (converges to) the true solution.
<hr>

### The QR-iteration method


An obvious problem with the Power Method is that we can only find one eigenvalue and one eigenvector at a time. The most common algorithm for computing eigenvalues and eigenvectors is the <i>QR-iteration</i> outlined below:
- Start with finding the QR-decomposition: $A=QR$
- Flip the order around and calculate a new matrix: $A_1= RQ$
- Find the QR-decomposition of the new matrix: $A_1=Q_1R_1$
- Flip the order again: $A_2=R_1Q_1$
- And so forth until result is good enough

Almost magically, the eigenvalues can be found on the main diagonal in the matrix $A_k$ obtained at the $k$th iteration. As with the Power Method, this is also an iterative method which converges to the true solution after a sufficient number of iterations.

_2.1) Implement the QR-iteration using a for-loop (based on the outline above). Then, run the algorithm for different number of iterations $N$. Compare the diagonal entries of the resulting matrix $A_N$ with the eigenvalues you computed in Task 1.1). Does the QR-iteration find all eigenvalues of $A$?_

Note that the matrix approaches an upper triangular matrix (the elements under the main diagonal gets smaller as you increase the number of iterations). Remember that the eigenvalues of a triangular matrix can be found on the main diagonal.

In [3]:
# 2.1) Implement the QR-iteration using a for-loop (based on the outline above). Then, run the algorithm for different number of iterations  N . Compare the diagonal entries of the resulting matrix  AN  with the eigenvalues you computed in Task 1.1). Does the QR-iteration find all eigenvalues of  A ?

import numpy as np

A_qr = A.copy() # Use a copy of A to avoid modifying the original matrix
N_qr_iterations = 100 # Number of iterations for QR-iteration

print(f"Initial Matrix A:\n{A_qr}\n")

for i in range(N_qr_iterations):
    Q, R = np.linalg.qr(A_qr) # QR decomposition: A = Q * R
    A_qr = R @ Q # Form new matrix: A_new = R * Q

    # Optional: print progress of diagonal elements
    # if i % 10 == 0 or i == N_qr_iterations - 1:
    #    print(f"Iteration {i+1}, Diagonal elements: {np.diag(A_qr)}")

print(f"\nQR-iteration Result after {N_qr_iterations} iterations:")
print(f"Final matrix A_N:\n{A_qr}")

qr_eigenvalues = np.diag(A_qr)
print(f"\nEigenvalues from QR-iteration (diagonal elements):\n{np.sort(qr_eigenvalues)[::-1]} # Sorted in descending order")

print(f"\nEigenvalues from NumPy (from Task 1.1):\n{np.sort(eigenvalues)[::-1]} # Sorted in descending order")

# Compare if the QR-iteration found all eigenvalues
# We'll compare the sorted eigenvalues for easier analysis

# Calculate the difference for comparison
diff = np.linalg.norm(np.sort(qr_eigenvalues) - np.sort(eigenvalues))

print(f"\nDifference between QR-iteration eigenvalues and NumPy eigenvalues: {diff:.6f}")

if diff < 1e-6:
    print("Conclusion: The QR-iteration appears to find all eigenvalues of A, matching NumPy's results closely.")
else:
    print("Conclusion: The QR-iteration did not find all eigenvalues of A, or did not converge sufficiently.")


Initial Matrix A:
[[1 2 0]
 [1 1 2]
 [1 3 1]]


QR-iteration Result after 100 iterations:
Final matrix A_N:
[[ 4.05137424e+00  1.23090532e+00 -8.08884564e-01]
 [ 6.21154972e-43 -1.53407020e+00  9.11360446e-01]
 [ 2.21888649e-92  1.66439197e-50  4.82695955e-01]]

Eigenvalues from QR-iteration (diagonal elements):
[ 4.05137424  0.48269595 -1.5340702 ] # Sorted in descending order

Eigenvalues from NumPy (from Task 1.1):
[ 4.05137424  0.48269595 -1.5340702 ] # Sorted in descending order

Difference between QR-iteration eigenvalues and NumPy eigenvalues: 0.000000
Conclusion: The QR-iteration appears to find all eigenvalues of A, matching NumPy's results closely.


_2.2) Below is the the QR-iteration defined as a function (starting from the input matrix A, and running until convergence). Call the function with_
- The same matrix $A$ as before
- A symmetric matrix, for example $B=A^TA$. Compare the result with the Spectral Theorem, i.e. the resulting matrix should be roughly
$$
\Lambda= \left( \begin{array}{ccc}
\lambda_1 &  & 0 \\
          & \ddots & \\
        0  &        & \lambda_n
          \end{array} \right)
          $$

**Caution:** The iterative process involves estimating the error at each iteration by comparing the diagonal elements of $A_k$ (approximated eigenvalues) with those of $A_{k-1}$. The iteration terminates if the maximum error falls below a specified tolerance (`tol`). It is crucial to note that this criterion is applicable only when all eigenvalues of $A$ are real (upper triangular Schur form), such as in the case of symmetric matrices. For complex eigenvalues this criterion may not provide reliable convergence assessment.

In [4]:
def QRiteration(A0, tol=1.0e-8):
    # Eigenvalue computation, QR-iteration
    # Default error tolerance in max-norm is 1.0e-8

    if tol<1.0e-15:   # Tol cant be smaller than machine epilon
        tol = 1.0e-14

    err = 1.0
    while err > tol:
        Q1, R1 = np.linalg.qr(A0)
        A1 = R1@Q1
        A1diag = np.diag(A1)   # Get A1s main diagonal
        A0diag = np.diag(A0)   # Get A0s main diagonal
        err = (np.linalg.norm(A1diag-A0diag,np.inf))/np.linalg.norm(A1diag,np.inf) # rel. error in max-norm
        A0 = A1

    return A1

**Some notes on the QR-method we have implemented today:**

- The QR-iteration method above only computes the eigenvalues, but it is possible to also the get the eigenvectors (we will not talk about this today, however). <br>
- The QR-iteration method is very expensive due the the QR-decomposition in each iteration. In the standard method the matrix can be transformed to a **Hessenberg** form before the loop, and that reduces the cost significantly. Also the convergence rate can be improved (leading to fewer iteration). This makes the practical QR-method efficient.<br>

<hr>